# 🤟 Qyran — Sign Language Recognition Model v3

**Architecture:** Conv1D → Bidirectional LSTM → Dense
**Input:** (30 frames × 126 landmarks) from MediaPipe Hands
**Strategy:** Top-50 classes + Real-time augmentation + TF.js export

### Перед запуском:
1. Загрузите в корень Google Drive: `landmarks.zip`, `label_map.json`, `annotations.csv`
2. Runtime → Change runtime type → **T4 GPU**
3. Запускайте все ячейки по порядку (Ctrl+F9)

## 1. Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive'
for f in ['landmarks.zip', 'label_map.json', 'annotations.csv']:
    assert os.path.exists(f'{DRIVE}/{f}'), f'❌ {f} не найден на Google Drive!'
print('✅ Все файлы найдены на Drive')

!unzip -q -o {DRIVE}/landmarks.zip -d /content/
train_n = len(os.listdir('/content/landmarks/train'))
test_n = len(os.listdir('/content/landmarks/test'))
print(f'✅ Распаковано: train={train_n}, test={test_n}')

## 2. Load Data — Top-50 Classes

In [ ]:
import numpy as np
import pandas as pd
import json

df = pd.read_csv(f'{DRIVE}/annotations.csv', sep='\t')

# Топ-50 классов по количеству train видео
TOP_N = 50
top_classes = df[df['train']==True]['text'].value_counts().head(TOP_N).index.tolist()
label_map = {cls: i for i, cls in enumerate(top_classes)}
idx_to_label = {i: cls for cls, i in label_map.items()}

SEQ_LEN = 30
FEATURES = 126
NUM_CLASSES = len(label_map)

def load_split(name, is_train):
    sdf = df[(df['train']==is_train) & (df['text'].isin(top_classes))]
    X, y, skip = [], [], 0
    for _, r in sdf.iterrows():
        p = f'/content/landmarks/{name}/{r["attachment_id"]}.npy'
        if os.path.exists(p):
            d = np.load(p)
            if d.shape == (SEQ_LEN, FEATURES):
                X.append(d); y.append(label_map[r['text']])
            else: skip += 1
        else: skip += 1
    print(f'  {name}: loaded {len(X)}, skipped {skip}')
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

X_train_raw, y_train_raw = load_split('train', True)
X_test, y_test = load_split('test', False)

print(f'\n✅ Classes: {NUM_CLASSES}')
print(f'✅ Train: {X_train_raw.shape} | Test: {X_test.shape}')
print(f'\nТоп-10 классов:')
for i, cls in enumerate(top_classes[:10]):
    cnt = (y_train_raw == i).sum()
    print(f'  {i+1}. "{cls}" — {cnt} samples')

## 3. Real-time Data Augmentation

In [ ]:
import tensorflow as tf

class LandmarkAugmentor(tf.keras.utils.Sequence):
    """
    Real-time augmentation generator.
    On each epoch, generates fresh augmented samples.
    
    Augmentations:
      1. Gaussian noise (σ=0.005-0.015) — устойчивость к шуму камеры
      2. Scale jitter (0.85-1.15) — разные расстояния до камеры  
      3. Temporal shift (±3 frames) — небольшие сдвиги по времени
      4. Hand mirror + swap — зеркальное отражение
      5. Speed perturbation — сжатие/растяжение по времени
    """
    def __init__(self, X, y, batch_size=64, augment=True):
        self.X = X
        self.y = y
        self.batch_size = batch_size
        self.augment = augment
        self.indices = np.arange(len(X))
        self.on_epoch_end()
    
    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))
    
    def on_epoch_end(self):
        np.random.shuffle(self.indices)
    
    def __getitem__(self, idx):
        batch_idx = self.indices[idx*self.batch_size : (idx+1)*self.batch_size]
        X_batch = self.X[batch_idx].copy()
        y_batch = self.y[batch_idx]
        
        if self.augment:
            for i in range(len(X_batch)):
                X_batch[i] = self._augment_sample(X_batch[i])
        
        return X_batch, y_batch
    
    def _augment_sample(self, seq):
        # 1. Gaussian noise
        if np.random.random() < 0.8:
            sigma = np.random.uniform(0.005, 0.015)
            seq = seq + np.random.normal(0, sigma, seq.shape).astype(np.float32)
        
        # 2. Scale jitter (разные расстояния до камеры)
        if np.random.random() < 0.7:
            scale = np.random.uniform(0.85, 1.15)
            seq = seq * scale
        
        # 3. Temporal shift
        if np.random.random() < 0.5:
            shift = np.random.randint(-3, 4)
            if shift != 0:
                seq = np.roll(seq, shift, axis=0)
                if shift > 0: seq[:shift] = 0
                else: seq[shift:] = 0
        
        # 4. Mirror X coords + swap hands
        if np.random.random() < 0.5:
            seq[:, 0::3] = 1.0 - seq[:, 0::3]  # flip X
            left = seq[:, :63].copy()
            seq[:, :63] = seq[:, 63:126]
            seq[:, 63:126] = left
        
        # 5. Speed perturbation (time stretch/compress)
        if np.random.random() < 0.4:
            speed = np.random.uniform(0.8, 1.2)
            n_frames = seq.shape[0]
            new_len = int(n_frames * speed)
            if new_len > 2:
                indices_old = np.linspace(0, n_frames-1, new_len).astype(int)
                stretched = seq[indices_old]
                # Resample back to original length
                indices_new = np.linspace(0, len(stretched)-1, n_frames).astype(int)
                seq = stretched[indices_new]
        
        return seq

train_gen = LandmarkAugmentor(X_train_raw, y_train_raw, batch_size=64, augment=True)
val_gen = LandmarkAugmentor(X_test, y_test, batch_size=64, augment=False)

# Проверка
X_sample, y_sample = train_gen[0]
print(f'✅ Batch shape: X={X_sample.shape}, y={y_sample.shape}')
print(f'✅ Augmentor ready: {len(train_gen)} batches/epoch')

## 4. Model Architecture — Conv1D + BiLSTM

In [ ]:
from tensorflow.keras import layers, models

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

def build_model(seq_len, features, num_classes):
    """
    Conv1D + Bidirectional LSTM model.
    
    All layers have explicit unique names to avoid
    gamma/beta naming conflicts in TF.js export.
    """
    inp = layers.Input(shape=(seq_len, features), name='input_landmarks')
    
    # === Feature extraction (Conv1D) ===
    x = layers.Conv1D(64, kernel_size=3, padding='same', activation='relu', name='conv1d_1')(inp)
    x = layers.BatchNormalization(name='bn_conv1')(x)
    x = layers.Dropout(0.2, name='drop_conv1')(x)
    
    x = layers.Conv1D(128, kernel_size=3, padding='same', activation='relu', name='conv1d_2')(x)
    x = layers.BatchNormalization(name='bn_conv2')(x)
    x = layers.Dropout(0.2, name='drop_conv2')(x)
    
    # === Temporal modeling (BiLSTM) ===
    x = layers.Bidirectional(
        layers.LSTM(256, return_sequences=True, dropout=0.25, recurrent_dropout=0.1, name='lstm_1'),
        name='bilstm_1'
    )(x)
    x = layers.BatchNormalization(name='bn_lstm1')(x)
    
    x = layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, dropout=0.25, recurrent_dropout=0.1, name='lstm_2'),
        name='bilstm_2'
    )(x)
    x = layers.BatchNormalization(name='bn_lstm2')(x)
    
    x = layers.Bidirectional(
        layers.LSTM(64, return_sequences=False, dropout=0.2, name='lstm_3'),
        name='bilstm_3'
    )(x)
    x = layers.BatchNormalization(name='bn_lstm3')(x)
    
    # === Classification head ===
    x = layers.Dense(256, activation='relu', name='dense_1')(x)
    x = layers.Dropout(0.3, name='drop_dense1')(x)
    x = layers.Dense(128, activation='relu', name='dense_2')(x)
    x = layers.Dropout(0.25, name='drop_dense2')(x)
    
    out = layers.Dense(num_classes, activation='softmax', name='output_classes')(x)
    
    model = models.Model(inp, out, name='qyran_sign_model')
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model(SEQ_LEN, FEATURES, NUM_CLASSES)
model.summary()

# Проверяем уникальность имён слоёв
names = [l.name for l in model.layers]
assert len(names) == len(set(names)), '❌ Duplicate layer names!'
print(f'\n✅ All {len(names)} layer names are unique — TF.js safe')

## 5. Training

In [ ]:
from tensorflow.keras import callbacks as cb

EPOCHS = 100

callbacks_list = [
    cb.ModelCheckpoint(
        '/content/best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    cb.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    cb.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks_list,
    verbose=1
)

print(f'\n🎯 Training complete!')

## 6. Evaluation & Metrics

In [ ]:
import matplotlib.pyplot as plt

# === Test Accuracy ===
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'🎯 Test Accuracy: {test_acc*100:.2f}%')
print(f'📉 Test Loss:     {test_loss:.4f}')

# === Top-5 Accuracy ===
y_probs = model.predict(X_test, verbose=0)
y_pred = y_probs.argmax(axis=1)
top5_correct = sum(y_test[i] in np.argsort(y_probs[i])[-5:] for i in range(len(y_test)))
print(f'🎯 Top-5 Accuracy: {top5_correct/len(y_test)*100:.2f}%')

# === Training Curves ===
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Val', linewidth=2)
ax1.set_title('Accuracy', fontsize=14)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'], label='Train', linewidth=2)
ax2.plot(history.history['val_loss'], label='Val', linewidth=2)
ax2.set_title('Loss', fontsize=14)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()

In [ ]:
from sklearn.metrics import classification_report

target_names = [idx_to_label[i] for i in range(NUM_CLASSES)]
print('📊 Classification Report:')
print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))

# Per-class accuracy (best & worst)
print('\n🏆 Лучшие 10 классов:')
per_class = {}
for i in range(NUM_CLASSES):
    mask = y_test == i
    if mask.sum() > 0:
        acc = (y_pred[mask] == i).mean()
        per_class[idx_to_label[i]] = acc

sorted_classes = sorted(per_class.items(), key=lambda x: x[1], reverse=True)
for name, acc in sorted_classes[:10]:
    print(f'  ✅ {name}: {acc*100:.1f}%')

print('\n⚠️ Худшие 10 классов:')
for name, acc in sorted_classes[-10:]:
    print(f'  ❌ {name}: {acc*100:.1f}%')

## 7. Save to Google Drive

In [ ]:
import shutil

SAVE_DIR = f'{DRIVE}/qyran_model_v3'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save models
model.save(f'{SAVE_DIR}/final_model.keras')
shutil.copy('/content/best_model.keras', f'{SAVE_DIR}/best_model.keras')
shutil.copy('/content/training_curves.png', f'{SAVE_DIR}/training_curves.png')

# Save label map (top-50)
with open(f'{SAVE_DIR}/label_map.json', 'w', encoding='utf-8') as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)

# Save config
config = {
    'num_classes': NUM_CLASSES,
    'sequence_length': SEQ_LEN,
    'features_per_frame': FEATURES,
    'top_n_classes': TOP_N,
    'test_accuracy': float(test_acc),
    'top5_accuracy': float(top5_correct / len(y_test)),
    'classes': top_classes,
    'tf_version': tf.__version__
}
with open(f'{SAVE_DIR}/config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print(f'✅ Saved to: {SAVE_DIR}')
for f in sorted(os.listdir(SAVE_DIR)):
    sz = os.path.getsize(f'{SAVE_DIR}/{f}')
    print(f'  📄 {f} ({sz/1024:.1f} KB)')

## 8. Export to TensorFlow.js

In [ ]:
!pip install tensorflowjs -q 2>/dev/null

import subprocess, shutil

TFJS_DIR = f'{SAVE_DIR}/tfjs_model'

# Шаг 1: Сохраняем как SavedModel (надёжнее для конвертации)
SAVED_MODEL_DIR = '/content/saved_model_tmp'
model.export(SAVED_MODEL_DIR)

# Шаг 2: Конвертируем через CLI
# Используем graph_model — он стабильнее с Keras 3
result = subprocess.run([
    'tensorflowjs_converter',
    '--input_format=tf_saved_model',
    '--output_format=tfjs_graph_model',
    '--signature_name=serving_default',
    '--saved_model_tags=serve',
    SAVED_MODEL_DIR,
    TFJS_DIR
], capture_output=True, text=True)

if result.returncode != 0:
    print('⚠️ Graph model failed, trying layers model...')
    # Фоллбэк: keras → layers model
    model.save('/content/model_for_tfjs.keras')
    result = subprocess.run([
        'tensorflowjs_converter',
        '--input_format=keras',
        '--output_format=tfjs_layers_model',
        '/content/model_for_tfjs.keras',
        TFJS_DIR
    ], capture_output=True, text=True)

if result.returncode == 0:
    # Копируем label_map рядом
    shutil.copy(f'{SAVE_DIR}/label_map.json', f'{TFJS_DIR}/label_map.json')
    shutil.copy(f'{SAVE_DIR}/config.json', f'{TFJS_DIR}/config.json')
    
    print(f'\n✅ TF.js model saved to: {TFJS_DIR}')
    for f in sorted(os.listdir(TFJS_DIR)):
        sz = os.path.getsize(f'{TFJS_DIR}/{f}')
        print(f'  📄 {f} ({sz/1024:.1f} KB)')
    print('\n📦 Скачайте папку tfjs_model и положите в public/model/ веб-приложения')
else:
    print(f'❌ Conversion failed:\n{result.stderr}')

In [ ]:
# Верификация: загрузим TF.js модель обратно и проверим
import tensorflowjs as tfjs

# Проверяем model.json
with open(f'{TFJS_DIR}/model.json', 'r') as f:
    mj = json.load(f)

model_format = mj.get('format', mj.get('modelTopology', {}).get('model_config', {}).get('class_name', 'unknown'))
print(f'Model format: {model_format}')

weights = mj.get('weightsManifest', [{}])[0].get('weights', [])
print(f'Weight entries: {len(weights)}')

# Проверяем нет ли дублей в именах весов
w_names = [w['name'] for w in weights]
dupes = [n for n in w_names if w_names.count(n) > 1]
if dupes:
    print(f'⚠️ Duplicate weight names: {set(dupes)}')
else:
    print(f'✅ All {len(w_names)} weight names unique — browser-safe!')

total_kb = sum(os.path.getsize(f'{TFJS_DIR}/{f}') for f in os.listdir(TFJS_DIR)) / 1024
print(f'📦 Total size: {total_kb:.0f} KB ({total_kb/1024:.1f} MB)')

## 9. Готово! 🎉

### Следующие шаги:
1. Скачайте папку `qyran_model_v3/tfjs_model/` с Google Drive
2. Положите её в `public/model/` вашего React-приложения
3. Используйте код из `useSignModel.ts` для загрузки и инференса